### 1. Setup Environment
Clone the repository and install requirements.

In [ ]:
# Remove old runs
!cd /content
!rm -rf robotic_affordance_project

# Clone the repo
!git clone https://github.com/francesco-dorati/robotic_affordance_project.git
%cd robotic_affordance_project

# Add the current directory to sys.path so 'import config' works immediately
import sys
import os
sys.path.append(os.getcwd()) 

!pip install -r requirements.txt

### 2. Mount Google Drive & Extract Data
Make sure your UMD .tar.gz files are uploaded to a folder named 'UMD_Dataset' in your Google Drive.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Create the destination folder
!mkdir -p data/raw

# 3. Copy with Progress Bar
!echo "Copying Tools Dataset..."
!rsync -ah --progress /content/drive/MyDrive/UMD_Dataset/Affordance\ Dataset\ Tools.tar.gz data/raw/

!echo "Copying Clutter Dataset..."
!rsync -ah --progress /content/drive/MyDrive/UMD_Dataset/Affordance\ Dataset\ Clutter.tar.gz data/raw/

# 4. Extract with File List
# Adding the 'v' (verbose) flag to -xf shows every file as it's unpacked.
# Note: This will create a very long output list!
!echo "Extracting Datasets..."
!tar -xvf data/raw/Affordance\ Dataset\ Tools.tar.gz -C data/raw/
!tar -xvf data/raw/Affordance\ Dataset\ Clutter.tar.gz -C data/raw/

# 5. VERIFICATION: This should show the 'tools' folder
!echo "Verifying..."
![ "$(ls -A data/raw/part-affordance-dataset/tools/knife_01)" ] && echo "Done!" || echo "Something went wrong..."

### 3. Precompute Normals

In [ ]:
import config
from utils.geometry import batch_precompute_normals

# 2. Check if folders exist and run the processing
# We use the paths defined in your config.py
raw_path = str(config.RAW_TOOLS)
processed_path = str(config.PROCESSED_TOOLS)

# Create the processed directory if it doesn't exist
os.makedirs(processed_path, exist_ok=True)

print(f"Reading from: {raw_path}")
print(f"Saving to:    {processed_path}")

# 3. Run the heavy lifting!
batch_precompute_normals(raw_path, processed_path)

print("--- Precomputation Complete! ---")

### 4. Start the Training Loop
Run the training script using the cloud GPU.

In [ ]:
!python scripts/03_train.py